# UrbanGap: contextual population and final validation

Review resident population within centre buffers and inspect the final dataset contract. Inputs: authoritative features, schema and validation reports. Outputs: checks, a diagnostic plot and feature roles. Implementation: `src/urbangap/population.py` and `validation.py`.

In [ ]:
from pathlib import Path
import sys

# Works when the kernel starts in the project root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from urbangap.common import INTERIM, PROCESSED, CRS

assert (PROCESSED / "sydney_hex_features.gpkg").exists(), "Run python scripts/run_pipeline.py first."


## Context extends beyond the prediction boundary

Buffers include contributions from all intersecting original SA1s, not only the selected study SA1s. Denominators remain complete original areas. Overlapping buffers share residents, so summing buffer populations does not estimate a distinct regional population. GeoPandas buffers preserve the historical circle approximation.

In [ ]:
features = gpd.read_file(PROCESSED / "sydney_hex_features.gpkg")
from urbangap.validation import validate_features
validate_features(features)
display(features[["population_est", "population_250m", "population_500m"]].describe())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(features.population_250m, features.population_500m, s=2, alpha=0.25)
ax.set(xlabel="Estimated residents within 250 m", ylabel="Estimated residents within 500 m", title="Nested population buffers")
plt.show()

## Final schema and regression evidence

`hex_id` is the identifier, `toilet_count` is the placement proxy, and geometry is for mapping. The explicit model feature list excludes old park representative-point counts. No model is trained here. Hold out spatial regions during later model evaluation and do not interpret absent toilets as proven unmet demand.

In [ ]:
import json
schema = json.loads((PROCESSED / "feature_schema.json").read_text())
report = json.loads((ROOT / "reports/validation/validation.json").read_text())
print("Validation:", report["status"])
print("Model feature columns:", len(schema["model_features"]))
display(pd.read_csv(ROOT / "reports/validation/feature_comparison.csv"))